# Chapter 16 &mdash; $P$-time and NP-time Defined via DTMs and NDTMs

**Concept 3 of the Chapter 16 decomposition:** *$P$-time and NP-time Defined via DTMs and NDTMs*

$P$: steps along a DTM's single path; NP: the <i>maximum</i> steps along any NDTM path.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-P-And-NP-Via-Machines/Concept-P-And-NP-Via-Machines.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The machine definitions, stated carefully because the NP one has a trap.

* $L \in P$ if some **DTM** decides $L$ in $O(n^k)$ steps for some constant $k$. One
  path, count its length.
* $L \in NP$ if some **NDTM** decides $L$ in $O(n^k)$ steps, where the cost of an NDTM
  is the **length of the longest path in its computation tree** &mdash; *not* the total
  number of nodes.

That is the trap. An NDTM's tree can have exponentially many nodes while every path is
short. NP-time measures **depth**, not **work**.

Equivalently (Concept 4): guess a polynomial-length certificate, then check it
deterministically in polynomial time. The guess is the branching; the check is the
path.

## 2. Definitions

### A computation tree, and its two measures

In [ ]:
def tree_stats(branching, depth):
    nodes = sum(branching ** d for d in range(depth + 1))
    return dict(longest_path=depth, total_nodes=nodes)

### A nondeterministic search, and its deterministic simulation

In [ ]:
from itertools import product
def nd_subset_sum(nums, target):
    # ONE path of the NDTM: guess a subset, then add it up.
    # Path length is O(n); the TREE has 2^n leaves.
    return dict(path_length=len(nums), tree_leaves=2 ** len(nums))

def det_subset_sum(nums, target):
    steps = 0
    for bits in product([0, 1], repeat=len(nums)):
        steps += len(nums)
        if sum(n for n, b in zip(nums, bits) if b) == target:
            return True, steps
    return False, steps

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;2.&nbsp;Corralling Problems: NP, NPC, and the Collapse Property](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Corralling-Problems/Concept-Corralling-Problems.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16-NPC/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;4.&nbsp;The Verifier View, and its Equivalence to the Decider View](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Verifier-View/Concept-Verifier-View.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Depth versus nodes.** The tree explodes; the paths do not.

In [ ]:
print("%-8s %-16s %s" % ("depth", "longest path", "total nodes"))
for d in [4, 8, 16, 30]:
    s = tree_stats(2, d)
    print("%-8d %-16d %s" % (d, s['longest_path'], format(s['total_nodes'], ',')))
s = tree_stats(2, 30)
assert s['longest_path'] == 30 and s['total_nodes'] > 10 ** 9
print("\nNP-time for that machine is 30, not two billion.")

Subset sum: the NDTM path is linear, the deterministic search is exponential.

In [ ]:
nums = [3, 34, 4, 12, 5, 2]
print("nondeterministic :", nd_subset_sum(nums, 9))
ok, steps = det_subset_sum(nums, 9)
print("deterministic    : found=%s after %d elementary steps" % (ok, steps))
assert ok

The gap grows with $n$.

In [ ]:
print("%-6s %-14s %s" % ("n", "NDTM path", "DTM steps (worst case)"))
for n in [4, 8, 12, 16]:
    nums = list(range(1, n + 1))
    target = sum(nums) + 1                      # unreachable: forces full search
    _, steps = det_subset_sum(nums, target)
    print("%-6d %-14d %s" % (n, n, format(steps, ',')))

So the definitions, side by side.

In [ ]:
print("P  : exists DTM  M, constant k, with M deciding L in O(n^k) STEPS")
print("NP : exists NDTM N, constant k, with N deciding L and every path")
print("     in its computation tree of length O(n^k)")
print()
print("The NDTM is allowed exponentially many paths.  It is not allowed a")
print("single LONG one.")

And why simulation costs exponential time but not exponential *depth*.

In [ ]:
for n in [10, 20, 30]:
    print("  n=%2d : NDTM depth %2d, DTM simulation visits up to %s nodes"
          % (n, n, format(2 ** n, ',')))
print("\nThat is the best known general simulation -- and improving it")
print("to polynomial would prove P = NP.")

## 4. Exercises


1. Why is it wrong to define NP-time as the number of nodes in the tree?
2. Show $P \subseteq NP$ from these definitions.
3. What is the best known deterministic simulation of an NDTM? What does it cost?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16-NPC/Concept-P-And-NP-Via-Machines')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')